In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
from sklearn.metrics import classification_report, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor
from imblearn.over_sampling import SMOTE

In [20]:
metabolic_data = pd.read_csv('/content/sample_data/ritual_food_metabolic_dataset.csv')

In [21]:
metabolic_data.head()

,age,bmi,bmi_category,diabetes_status,fasting_state,festival,region,food_name,glycemic_index,carbs_per_item_g,sugar_per_item_g,glucose_spike_risk,safe_sugar_limit_g,safe_portion_count
0,65,27.7,overweight,1,0,Christmas,Kerala,Plum Cake,58,27,15,high,12.92,0.63
1,19,18.6,normal,1,1,Diwali,North India,Besan Ladoo,50,22,12,high,9.10,0.15
2,26,24.5,normal,1,1,Pongal,Tamil Nadu,Sakkarai Pongal,70,30,15,high,12.37,0.41
3,26,32.6,obese,0,0,Christmas,Kerala,Plum Cake,58,27,15,moderate,21.76,0.96
4,75,39.1,obese,0,0,Onam,Kerala,Payasam,65,28,14,high,22.25,0.89


In [22]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

cat_cols = ['bmi_category', 'festival', 'region', 'food_name']

for col in cat_cols:
    le = LabelEncoder()
    metabolic_data[col] = le.fit_transform(metabolic_data[col])
    encoders[col] = le   # store encoder for this column


In [23]:
# Separate encoder for target labels
risk_encoder = LabelEncoder()
metabolic_data['risk_encoded'] = risk_encoder.fit_transform(
    metabolic_data['glucose_spike_risk']
)


lets prepare our Input and target labels now! for different models!

In [24]:
x = metabolic_data[['age', 'bmi', 'diabetes_status', 'fasting_state',
    'bmi_category',  'festival', 'region', 'food_name',
    'glycemic_index', 'carbs_per_item_g', 'sugar_per_item_g']]

y_class = metabolic_data[['risk_encoded']]

y_reg = metabolic_data[['safe_portion_count']]



Now lets make our train test set

In [25]:
x_train,x_test,y_class_train,y_class_test,y_reg_train,y_reg_test = train_test_split(x,y_class,y_reg, test_size=0.2, random_state=42)

Now lets call our models!!


In [26]:
classifier = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.6,
    colsample_bytree=0.6,
    random_state=42,
    eval_metric='mlogloss'
)

In [27]:
regressor = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [28]:
# now lets fit our tarining data

classifier.fit(x_train,y_class_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [29]:
regressor.fit(x_train,y_reg_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

lets finally evaluate our models!

In [30]:
y_class_pred = classifier.predict(x_test)

print(classification_report(y_class_pred,y_class_test))

              precision    recall  f1-score   support

           0       0.71      0.58      0.64      1048
           1       0.26      0.58      0.36        57
           2       0.58      0.58      0.58       630
           3       0.47      0.68      0.55       265

    accuracy                           0.59      2000
   macro avg       0.50      0.60      0.53      2000
weighted avg       0.62      0.59      0.60      2000



In [31]:

y_reg_pred = regressor.predict(x_test)

mae = mean_absolute_error(y_reg_test, y_reg_pred)
print('MAE (safe portion):', mae)

MAE (safe portion): 0.1732921451330185


In [32]:
joblib.dump(classifier, 'final_risk_classifier.pkl')
joblib.dump(regressor, 'final_portion_regressor.pkl')
joblib.dump(risk_encoder, 'risk_label_encoder.pkl')
joblib.dump(encoders, 'categorical_encoders.pkl')

['categorical_encoders.pkl']

In [38]:
import joblib
import pandas as pd

# Loading Saved models
risk_model = joblib.load("/content/final_risk_classifier.pkl")
portion_model = joblib.load("/content/final_portion_regressor.pkl")
encoders = joblib.load("/content/categorical_encoders.pkl")
risk_encoder = joblib.load("/content/risk_label_encoder.pkl")

print(" Models loaded successfully")


 Models loaded successfully


In [39]:
sample = {
    "age": 45,
    "bmi": 27.5,
    "diabetes_status": 1,
    "fasting_state": 0,
    "bmi_category": "overweight",
    "festival": "Ganesh Chaturthi",
    "region": "Maharashtra",
    "food_name": "Steamed Modak",
    "glycemic_index": 60,
    "carbs_per_item_g": 20,
    "sugar_per_item_g": 10,
}

df = pd.DataFrame([sample])
df


,age,bmi,diabetes_status,fasting_state,bmi_category,festival,region,food_name,glycemic_index,carbs_per_item_g,sugar_per_item_g
0,45,27.5,1,0,overweight,Ganesh Chaturthi,Maharashtra,Steamed Modak,60,20,10


In [40]:
for col, encoder in encoders.items():
    df[col] = encoder.transform(df[col])

df


,age,bmi,diabetes_status,fasting_state,bmi_category,festival,region,food_name,glycemic_index,carbs_per_item_g,sugar_per_item_g
0,45,27.5,1,0,2,3,2,8,60,20,10


In [36]:
risk_pred_num = risk_model.predict(df)[0]
portion_pred = portion_model.predict(df)[0]

risk_pred_label = risk_encoder.inverse_transform([risk_pred_num])[0]

print(" Predicted glucose risk:", risk_pred_label)
print(" Predicted safe portion:", round(portion_pred, 2), "servings")


 Predicted glucose risk: high
 Predicted safe portion: 0.77 servings
